# Prática de Transcriptômica: expressão diferencial com DESeq2

**Introdução à Bioinformática – PPGED/IME · Aula 4 (Transcriptômica)**

Nesta prática, vamos descobrir quais genes mudam de atividade quando células humanas são tratadas com um corticoide.

**Os dados (conjunto `airway`)**
- Estudo de Himes e colaboradores (PLoS One, 2014), dados públicos no GEO (GSE52778).
- Células de músculo liso das vias aéreas (o músculo que envolve os brônquios) de **4 doadores**.
- As células de cada doador foram divididas em duas partes: **controle** e **tratada com dexametasona** (1 µM por 18 h).
- As leituras já foram mapeadas com o STAR no genoma humano (Ensembl 75) e contadas por gene. Partimos da **tabela de contagens**.

**Antes de começar**
1. No Colab: *Ambiente de execução → Alterar tipo de ambiente de execução → R*.
2. Execute as células **em ordem**, de cima para baixo (Shift + Enter).
3. Não é preciso escrever código: o objetivo é **executar e interpretar** cada resultado.

**Roteiro**
1. Preparação do ambiente
2. Conhecendo os dados
3. Desenho experimental
4. Controle de qualidade (PCA)
5. Análise de expressão diferencial
6. Resultados: três perguntas diferentes
7. Visualização
8. Exportar os resultados
9. Exercícios

## 1. Preparação do ambiente

Instalamos o **BiocManager** (o gerenciador de pacotes do Bioconductor) e, com ele, os pacotes da aula:
- **airway**: os dados do experimento;
- **DESeq2**: a ferramenta de análise de expressão diferencial;
- **ggplot2**: pacote de gráficos (já vem junto com o DESeq2).

⏱️ A instalação leva de 5 a 10 minutos. As mensagens em vermelho durante a instalação são normais.

In [ ]:
# Instala o gerenciador de pacotes do Bioconductor e os pacotes da aula
install.packages("BiocManager")
BiocManager::install(c("airway", "DESeq2"))

library(airway)   # dados do experimento
library(DESeq2)   # análise de expressão diferencial
library(ggplot2)  # gráficos

## 2. Conhecendo os dados

O objeto `airway` guarda três peças juntas:
- **`colData`**: a ficha das amostras (doador, tratamento e códigos de catálogo do GEO e do SRA);
- **`assay`**: a tabela de contagens (genes nas linhas, amostras nas colunas);
- **`rowData`**: a ficha dos genes (nome, tipo, cromossomo, posição).

### 2.1 Ficha das amostras

Observe as colunas `cell` (doador) e `dex` (`untrt` = controle, `trt` = tratado). Cada doador aparece duas vezes: é um **desenho pareado**, com 4 réplicas biológicas por condição.

In [ ]:
data(airway)       # carrega o experimento
colData(airway)    # ficha das amostras: doador (cell) e tratamento (dex)

### 2.2 Tabela de contagens

Cada número é **quantas leituras caíram naquele gene, naquela amostra**. Os genes são identificados pelo código do Ensembl (ENSG...).

⚠️ Esses números **ainda não podem ser comparados diretamente**: as amostras foram sequenciadas com profundidades diferentes. Quem corrige isso é o DESeq2.

In [ ]:
head(assay(airway))   # primeiras linhas da tabela de contagens

### 2.3 Tamanho do experimento

- `dim`: quantos genes e quantas amostras;
- `colSums`: total de leituras de cada amostra, em milhões (repare que varia bastante);
- `table`: quantos genes de cada tipo (nem todo "gene" codifica proteína).

In [ ]:
dim(airway)                                         # genes × amostras
round(colSums(assay(airway)) / 1e6, 1)              # milhões de leituras por amostra
sort(table(rowData(airway)$gene_biotype), decreasing = TRUE)[1:8]   # tipos de gene mais comuns

## 3. Desenho experimental

Três passos:
1. Definimos **"não tratado"** como a referência. Assim, valores positivos de mudança significam "mais expresso no tratado".
2. Informamos o desenho `~ cell + dex`: o DESeq2 **desconta a diferença natural entre os doadores** e testa o efeito do tratamento. É como um ensaio de "antes e depois" no mesmo corpo de prova.
3. Removemos genes com menos de 10 leituras no total: eles não têm informação suficiente para o teste.

In [ ]:
# Define "não tratado" (untrt) como referência de comparação
airway$dex <- relevel(airway$dex, ref = "untrt")

# Desenho: considerar o doador (cell) e testar o efeito do tratamento (dex)
dds <- DESeqDataSet(airway, design = ~ cell + dex)

# Remove genes quase sem leituras (pouca informação)
dds <- dds[rowSums(counts(dds)) >= 10, ]

nrow(dds)   # quantos genes restaram

## 4. Controle de qualidade: PCA

Antes de procurar genes, verificamos se as amostras se comportam como esperado.

- **`vst`** transforma as contagens para uma escala parecida com a logarítmica, para que genes muito expressos não dominem o gráfico. Serve **só para visualização**.
- **PCA** resume os 500 genes que mais variam em dois eixos. **Cada ponto é uma amostra.** Pontos próximos = amostras com perfis parecidos.

**O que esperar:** o eixo horizontal (PC1) separa tratados e não tratados; o vertical (PC2) separa os doadores.

In [ ]:
vsd <- vst(dds)                              # transforma os dados para visualização
plotPCA(vsd, intgroup = c("dex", "cell"))    # versão padrão do DESeq2

### 4.1 Uma PCA mais fácil de ler

Mesma análise, agora com **cor = doador** e **formato = tratamento**.

In [ ]:
pca_df <- plotPCA(vsd, intgroup = c("dex", "cell"), returnData = TRUE)   # dados do gráfico
pv <- round(100 * attr(pca_df, "percentVar"))                             # % de variância de cada eixo

ggplot(pca_df, aes(x = PC1, y = PC2, color = cell, shape = dex)) +
  geom_point(size = 4) +
  labs(x = paste0("PC1: ", pv[1], "% da variância"),
       y = paste0("PC2: ", pv[2], "% da variância"),
       color = "Doador", shape = "Tratamento")

### 4.2 Quais genes formam os eixos?

Refazemos a mesma PCA "por dentro" para ver:
- quanto cada eixo explica (`summary`);
- quais genes têm mais **peso** no PC1, o eixo do tratamento.

Como os tratados ficaram à direita, genes com **peso positivo** tendem a estar **mais expressos nos tratados**; com peso negativo, menos expressos.

⚠️ Os pesos indicam **candidatos**, não a lista oficial de genes alterados. Essa vem do DESeq2 (seção 5).

In [ ]:
mat <- assay(vsd)                                     # dados transformados (genes × amostras)
variancia <- apply(mat, 1, var)                       # variância de cada gene
top500 <- order(variancia, decreasing = TRUE)[1:500]  # os 500 que mais variam

pca <- prcomp(t(mat[top500, ]))                       # a mesma PCA do gráfico
summary(pca)                                          # confira: PC1 ≈ 41%, PC2 ≈ 26%

In [ ]:
pesos_PC1 <- pca$rotation[, "PC1"]                    # peso de cada gene no PC1

top_pos <- head(sort(pesos_PC1, decreasing = TRUE), 10)   # puxam para o lado dos tratados
top_neg <- head(sort(pesos_PC1), 10)                      # puxam para o lado dos não tratados

data.frame(gene = rowData(vsd)[names(top_pos), "gene_name"], peso = round(top_pos, 3))
data.frame(gene = rowData(vsd)[names(top_neg), "gene_name"], peso = round(top_neg, 3))

### 4.3 (Opcional) PC3 e PC4: as diferenças entre doadores

Os 4 primeiros eixos explicam quase 97% da variação. Separar 4 pessoas exige até 3 eixos, então PC2, PC3 e PC4 provavelmente refletem os **doadores**. Se as duas amostras de cada doador ficarem juntas, a interpretação se confirma.

In [ ]:
plot(pca$x[, "PC3"], pca$x[, "PC4"], pch = 19, col = as.integer(vsd$cell),
     xlab = "PC3", ylab = "PC4", main = "PC3 × PC4")
text(pca$x[, "PC3"], pca$x[, "PC4"], labels = paste(vsd$cell, vsd$dex), pos = 3, cex = 0.7)

## 5. Análise de expressão diferencial

A função `DESeq` faz tudo de uma vez:
1. **Normalização** (`estimating size factors`): corrige a diferença no total de leituras entre amostras.
2. **Dispersão** (`estimating dispersions`): mede o "ruído" de cada gene entre réplicas, "emprestando informação" dos milhares de genes para compensar as poucas réplicas.
3. **Teste** (`fitting model and testing`): para cada gene, verifica se o tratamento mudou a expressão além do ruído.

O `summary` mostra o placar: quantos genes subiram e desceram com padj < 0,05.

In [ ]:
dds <- DESeq(dds)                  # normaliza, estima variação e testa cada gene
res <- results(dds, alpha = 0.05)  # resultados com corte de padj < 0,05
summary(res)                       # quantos genes subiram e quantos desceram

### 5.1 Fatores de normalização

Um número por amostra. Acima de 1 = amostra com mais leituras que a média; abaixo de 1 = com menos.

In [ ]:
round(sizeFactors(dds), 2)

## 6. Resultados: três perguntas diferentes

| Coluna | Significado |
|---|---|
| `baseMean` | expressão média do gene (normalizada) |
| `log2FoldChange` | quanto mudou, em log2: +1 = dobrou; −1 = caiu pela metade; +3 = 8× |
| `lfcSE` | incerteza (erro padrão) do log2FoldChange |
| `padj` | p-valor corrigido para milhares de testes; < 0,05 = confiável |

Vamos ordenar os genes de três formas, porque cada uma responde a uma pergunta diferente.

### 6.1 Quais mudanças são mais **certas**? (ordenado por padj)

In [ ]:
res_ord <- res[order(res$padj), ]               # do mais ao menos significativo

top_padj <- as.data.frame(head(res_ord, 10))
top_padj$gene <- rowData(dds)[rownames(top_padj), "gene_name"]
top_padj[, c("gene", "baseMean", "log2FoldChange", "lfcSE", "padj")]

### 6.2 Quais genes mudaram **mais**? (ordenado por log2FoldChange)

⚠️ Esta lista favorece genes **pouco expressos** (baseMean baixo): sair de 1 para 30 leituras já é "30 vezes". Repare no `lfcSE` alto desses genes, sinal de estimativa incerta.

In [ ]:
sig <- res[which(res$padj < 0.05), ]                                       # só os significativos
maiores <- as.data.frame(head(sig[order(sig$log2FoldChange, decreasing = TRUE), ], 10))
maiores$gene <- rowData(dds)[rownames(maiores), "gene_name"]
maiores[, c("gene", "baseMean", "log2FoldChange", "lfcSE", "padj")]

### 6.3 Quais mudaram mais **de forma confiável**? (log2FoldChange encolhido)

A função `lfcShrink` "encolhe" em direção a zero os log2FoldChange incertos e preserva os bem medidos. É a melhor lista para escolher **candidatos**.

⏱️ Instala o pacote `apeglm` (1 a 2 minutos na primeira vez).

In [ ]:
BiocManager::install("apeglm")
resultsNames(dds)                                   # nome do coeficiente: "dex_trt_vs_untrt"

res_shr <- lfcShrink(dds, coef = "dex_trt_vs_untrt", type = "apeglm")

sig_shr <- res_shr[which(res_shr$padj < 0.05), ]
top_shr <- as.data.frame(head(sig_shr[order(sig_shr$log2FoldChange, decreasing = TRUE), ], 10))
top_shr$gene <- rowData(dds)[rownames(top_shr), "gene_name"]
top_shr[, c("gene", "baseMean", "log2FoldChange", "lfcSE", "padj")]

### 6.4 Exigindo uma mudança mínima no teste

O critério padj < 0,05 inclui mudanças pequenas (10%, 20%). Com `lfcThreshold = 1`, o DESeq2 testa se o gene mudou **mais do que o dobro**. Compare o placar com o da seção 5.

In [ ]:
res_lfc1 <- results(dds, alpha = 0.05, lfcThreshold = 1)
summary(res_lfc1)

## 7. Visualização

- **MA plot**: expressão média (eixo X) × mudança (eixo Y). Pontos coloridos = significativos.
- **Volcano plot**: tamanho da mudança (eixo X) × certeza estatística (eixo Y). Em vermelho, genes com padj < 0,05 **e** mudança acima de 2×. Os 10 mais significativos aparecem com nome.

In [ ]:
plotMA(res, ylim = c(-5, 5))

In [ ]:
vol <- as.data.frame(res)
vol <- vol[!is.na(vol$padj), ]                                   # remove genes não testados
vol$gene <- rowData(dds)[rownames(vol), "gene_name"]
destaque <- vol$padj < 0.05 & abs(vol$log2FoldChange) > 1

plot(vol$log2FoldChange, -log10(vol$padj), pch = 20, cex = 0.6,
     col = ifelse(destaque, "red", "grey"),
     xlab = "log2 Fold Change", ylab = "-log10(padj)",
     main = "Genes alterados pela dexametasona")
abline(v = c(-1, 1), h = -log10(0.05), lty = 2)                  # limites: 2× e padj = 0,05

top <- head(vol[order(vol$padj), ], 10)                           # 10 mais significativos
text(top$log2FoldChange, -log10(top$padj), labels = top$gene, pos = 3, cex = 0.7)

## 8. Exportar os resultados

Gera uma planilha com todos os genes, já com o nome de cada um. Para baixar: no Colab, clique no ícone de pasta (barra lateral esquerda), depois nos três pontinhos ao lado de `resultados_DESeq2.csv` → *Fazer download*.

In [ ]:
tabela <- as.data.frame(res_ord)
tabela$gene <- rowData(dds)[rownames(tabela), "gene_name"]
tabela <- tabela[, c("gene", setdiff(names(tabela), "gene"))]    # nome do gene na 1ª coluna

write.csv(tabela, "resultados_DESeq2.csv")

## 9. Exercícios

1. Na seção 6.4, troque `lfcThreshold = 1` por `lfcThreshold = 2`. Quantos genes continuam significativos?
2. Compare as listas das seções 4.2 (pesos do PC1), 6.1 e 6.3. Quais genes aparecem em mais de uma? Por quê?
3. Escolha um gene da seção 6.3, pesquise o código ENSG em [ensembl.org](https://www.ensembl.org) e descreva sua função em 3 linhas.
4. **Desafio:** repita a análise com o pacote **edgeR** e compare os 10 primeiros genes com os do DESeq2.